In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

# ツール定義
def define_tools():
    #print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()


tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")



------define_tools(ツール定義)------


'質問:こんにちは！'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:東北6県は？'

東北6県は以下の通りです：

1. 青森県（あおもりけん）
2. 岩手県（いわてけん）
3. 宮城県（みやぎけん）
4. 秋田県（あきたけん）
5. 山形県（やまがたけん）
6. 福島県（ふくしまけん）


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産には、以下のようなものがあります。

1. **至福のまどろみ** - 特製の和三盆の大福や、色々な味のショコラが楽しめます。
   - [詳細はこちら](https://shunsentanbou.pref.miyagi.jp/theme/miyage/)

2. **宮城のお土産一覧** - 宮城に来たら欠かせないお土産を詳しく紹介しています。
   - [詳細はこちら](https://retty.me/area/PRE04/LCAT99/CAT991/)

3. **2025年の宮城の人気お土産20選** - 有名な地元の名産品が特集されており、訪れる人におすすめです。
   - [詳細はこちら](https://tsplus.asahi.co.jp/articles/gift/75865/)

4. **名物の紹介** - 宮城の特産品として、牛たんや仙台味噌などが有名です。
   - [詳細はこちら](https://www.fujisaki.co.jp/sendai-omiyage/miyagi-sendai-gentei.html)

5. **お土産のカテゴリー** - 宮城県内でのおすすめお土産品がまとまっているページです。
   - [詳細はこちら](https://miyagi.mytabi.net/cat.php?cat=souvenir)

これらは、宮城県を訪れた際に試してみる価値があるお土産の一部です。各リンクからさらに詳しい内容や購入方法を確認できますので、ぜひチェックしてみてください。

---ご利用ありがとうございました！---
